In [1]:
library(dplyr)
library(tidyr)
library(pheatmap)
library(viridis)
library(tools)
library(cluster)
library(igraph)
library(tibble)

library(clusterProfiler)
library(org.Hs.eg.db)
library(dplyr)

Warning message:
“程辑包‘dplyr’是用R版本4.2.3 来建造的”

载入程辑包：‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“程辑包‘tidyr’是用R版本4.2.3 来建造的”
Warning message:
“程辑包‘viridis’是用R版本4.2.3 来建造的”
载入需要的程辑包：viridisLite

Warning message:
“程辑包‘viridisLite’是用R版本4.2.3 来建造的”
Warning message:
“程辑包‘cluster’是用R版本4.2.3 来建造的”

载入程辑包：‘igraph’


The following object is masked from ‘package:tidyr’:

    crossing


The following objects are masked from ‘package:dplyr’:

    as_data_frame, groups, union


The following objects are masked from ‘package:stats’:

    decompose, spectrum


The following object is masked from ‘package:base’:

    union



载入程辑包：‘tibble’


The following object is masked from ‘package:igraph’:

    as_data_frame


Warning message:
“程辑包‘clusterProfiler’是用R版本4.2.2 来建造的”


Registered S3 methods overwritten by 'treeio':
  method              from    
  

In [2]:
# 读入
signature_Macro <- read.csv(
  "Example_refbuilder_Macro/ALL_cGEP_top_genes_Mono_Macro_wide_filt.csv",
  check.names = FALSE,
  stringsAsFactors = FALSE)
# 转成 list（核心）
cluster_genes <- lapply(signature_Macro, function(x) {
  x[!is.na(x) & x != ""]
})


gene_df <- data.frame(
  cGEP_Cluster = names(cluster_genes),  # 确保这里是 cGEP1, cGEP2...
  cGEP_signature = sapply(cluster_genes, function(x) paste(x, collapse = ",")),
  stringsAsFactors = FALSE
)
gene_df

,cGEP_Cluster,cGEP_signature
,<chr>,<chr>
cGEP1,cGEP1,"IL1B,PLAUR,C15ORF48,SRGN,SOD2,EREG,NAMPT,BCL2A1,CXCL8,CXCL2,G0S2,NFKBIA,BTG1,SAMSN1,AQP9,CXCL3,FTH1,TIMP1,OLR1,THBS1,PNRC1,CD44,IER3,NLRP3,C5AR1,CCL20,SAT1,TNFAIP3,SLC2A3,PTGS2"
cGEP2,cGEP2,"HSP90AA1,HSPA1A,HSPB1,HSPA1B,DNAJB1,HSPE1,IER5,ZFAND2A,HSPD1,HSPH1,HSPA6,DUSP1,RHOB,BAG3,DNAJB4,RGS2,UBC,MTRNR2L2,HSP90AB1,AP000769.1,ND3,CLK1,KLF2,DNAJA4,UBB,DNAJA1,CACYBP,RPS4Y1,SH3BGRL3,SHISAL2A"
cGEP3,cGEP3,"FABP7,ZIC5,RP11-408N14.1,RP1-56K13.2,FGG,SPRR1B,PRSS57,DPP10,HOXD10,RPS18,RPS6,SFTPA1,SPRR2D,SFTPA2,AP001171.1,RPS23,RPS4X,RPL3,RPS3,POU3F3,RPS12,CLDN2,RPLP0,RPS5,NPM1,FGB,RPS2,RPS7,RPL10A,RP11-817J15.2"
cGEP4,cGEP4,"DNAJB1,HSPA1B,HSPA1A,HSP90AA1,BAG3,HSPB1,HSPH1,HSP90AA2P,HSPD1,HSPA6,UBC,HSP90AB1,DNAJB1P1,JUN,DNAJA1,ZFAND2A,HSPE1,HSPA8,IER5,HSP90AA4P,DUSP1,DNAJA4,ZSCAN12P1,PCYT1B,PPP1R15A,ATF3,NR4A1,RGS2,UBB,HSPA7"
cGEP5,cGEP5,"S100A9,S100A8,LYZ,S100A12,VCAN,GAPDH,S100A6,AC020656.1,FCN1,TSPO,PLBD1,VIM,CSTA,METTL9,SELL,ATP5MPL,S100A4,TKT,LGALS1,RF00012,CD36,S100A10,RP11-1143G9.4,MGST1,RETN,SERPINB1,CES1,ATP5F1E,TALDO1,CDA"
cGEP6,cGEP6,"MTCO1P53,S100A9,RP11-240E2.2,S100A8,LYZ,FCN1,VCAN,S100A12,Metazoa-SRP,RP5-857K21.11,AC138744.2,RP4-635A23.6,AC016739.2,SELL,MNDA,S100A6,TKT,LA16C-306A4.2,RP11-417L19.5,4h.2,RP3-417G15.1,PLBD1,CSTA,CTD-2649C14.2,RP11-362F19.2,RP5-857K21.7,CTSS,TSPO,S100A4,CTB-181H17.1"
cGEP7,cGEP7,"AC079325.2,FCGR3A,LST1,C19ORF38,IFITM2,RPS19,RHOC,CNTN5,CDKN1C,C10ORF54,SMIM25,IFITM3,AC104809.2,AIF1,NDNF,C15ORF39,NAP1L1,AC104809.4,LINC01272,MS4A7,LOC100130520,TCF7L2,LINC02334,RP5-857K21.11,SERPINA1,COTL1,FCER1G,CD79B,LOC101928037,LOC100507639"
cGEP8,cGEP8,"GZMB,CLEC4C,LILRA4,CLIC3,SMPD3,PACSIN1,LRRC26,TSPAN13,PTCRA,MZB1,ITM2C,IGJ,IRF7,TLR9,JCHAIN,RP11-73G16.2,IRF4,SCT,FAM129C,LOC101928123,C12orf75,TCL1A,PPP1R14B,CXCR3,DERL3,SPIB,LOC101929219,RP1-302G2.5,TCF4,MAP1A"
cGEP9,cGEP9,"ACP5,LGALS3,CALR,CD63,PLA2G7,FTL,CTSB,CTSD,TGFBI,PRDX1,FPR3,CAPG,CTSL,S100A11,FABP5,NPC2,YBX1,PSAP,PMP22,FABP5P7,IFI30,HSP90B1,APOE,AC026366.1,PPIB,ATP6V1F,ANXA2,PCOLCE2,HLA-DQA2,HLA-DRB5"


In [3]:
#Macro_gep_anno <- read.csv('./3.1.Macro_GEP_Anno/3.3.Macro_GEP77_Anno_final.csv')
Macro_gep_anno <- read.csv('/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.6.cNMF_menchmark/cGEP_signature/Macrophage_cGEP77.csv')

Macro_gep_anno

cGEP_Cluster,cGEP_Anno_Name,Category
<chr>,<chr>,<chr>
cGEP26,Macro_M1_ISG,Functional
cGEP11,Doublet_T_Cell,Doublet
cGEP10,mregDC,Lineage
cGEP49,Macro_M1_Inflammatory,Functional
cGEP13,Macro_Osteoclast-like,Lineage
cGEP51,Macro_M1_Early_Activation,Functional
cGEP16,Macro_M2_LAM,Lineage
cGEP18,Macro_M2_Homeostasis,Lineage
cGEP44,Fibroblast,Doublet


In [4]:
final_combined_df <- Macro_gep_anno %>%
  left_join(gene_df, by = "cGEP_Cluster")
final_combined_df

cGEP_Cluster,cGEP_Anno_Name,Category,cGEP_signature
<chr>,<chr>,<chr>,<chr>
cGEP26,Macro_M1_ISG,Functional,"ISG15,GBP1,TNFSF10,IFIT3,CXCL10,IFIT1,MX1,CXCL11,IFITM3,IFIT2,IFI6,LY6E,HLA.B,EPSTI1,HLA.C,ISG20,RSAD2,IFI44L,LAP3,B2M,APOBEC3A,GBP4,CALHM6,IDO1,PPAPDC2,TYMP,IGLC1,STAT1,UBE2L6,OAS1"
cGEP11,Doublet_T_Cell,Doublet,"AC245427.1,IL32,CD3D,CD3E,PTPRCAP,TRAC,GZMA,CD2,TRBC2,CD3G,LCK,CCL5,LOC101929531,NKG7,CD7,TRBC1,GZMM,CTSW,KLRB1,CST7,SPOCK2,CD247,GZMH,CD27,AP002004.1,ETS1,GNLY,CD8A,PRF1,PCED1B-AS1"
cGEP10,mregDC,Lineage,"CCR7,FSCN1,LAMP3,BIRC3,LAD1,TXN,EBI3,CCL19,AL109741.1,MARCKSL1,IDO1,CCL22,AC007216.2,SLCO5A1,SINHCAF,HMSD,CD200,NUB1,GPR157,AOC1,TBC1D4,AC112719.2,LIMCH1,BX255923.3,RAB9A,PPA1,CSF2RA,DAPP1,BCL2L14,KIF2A"
cGEP49,Macro_M1_Inflammatory,Functional,"CCL3L3,CCL3,CCL4,IL1B,NFKBIA,CCL4L2,ZFP36,BCL2A1,CXCL3,TNF,PSME2,SOD2,CXCL2,IER3,KB-1507C5.4,PLAC8,CXCL8,SRGN,GADD45B,HLA-DRB1,HLA-DRA,GBP1,CH17-373J23.1,APOBEC3A,ATP5EP2,IFITM3,CD83,PPP1R15A,VAMP5,NCF1"
cGEP13,Macro_Osteoclast-like,Lineage,"CTSK,MMP9,CKB,SIGLEC15,SLC9B2,ACP5,AK5,NDRG4,CST3,AP000904.1,MT-RNR1,TCIRG1,THOP1,SPINK2,ANXA8L1,ATP6V0D2,S100A3,ITGB3,MYH16,RGS10,SLC22A1,HTRA3,DPP4,HSPB7,COL6A2,HYAL1,MTATP6P1,BAMBI,MT-RNR2,RP11-54O7.1"
cGEP51,Macro_M1_Early_Activation,Functional,"CCL3L1,CCL3,CCL3L3,CCL4,IL1B,CCL4L2,TNF,NFKBIA,CD83,PPP1R15A,ZFP36,JUN,GADD45B,FOS,IER2,IER3,ATF3,TNFAIP3,JUNB,KLF6,CXCL8,FOSB,RP3-393E18.2,NR4A1,DUSP2,IL1A,NFKBIZ,SOD2,SGK1,NR4A2"
cGEP16,Macro_M2_LAM,Lineage,"APOE,CTSD,GPNMB,APOC1,RP11-368J21.3,LGMN,CTSB,PSAP,C1QB,PLD3,ACP5,FTL,C1QC,LIPA,CD63,C1QA,PLA2G7,CD68,FUCA1,CAPG,PRDX1,RP11-696F12.1,ASAH1,GRN,CTSZ,TREM2,HLA-DRB4,LGALS3,CTSL,CTSC"
cGEP18,Macro_M2_Homeostasis,Lineage,"MTRNR2L5,C1QB,NPC2,ATXN8OS,CD74,NAPSB,C1QC,TREM2,PCED1B-AS1,CTC-471J1.9,AL353644.10,B2M,PSME2P2,AC004448.5,HLA-DPA1,C1QA,HLA-DRA,TMSB4X,RP11-680H20.1,ALOX5AP,HLA-DRB1,FCER1G,HLA-DMB,UBBP4,COX6A1P2,FYB1,ATP5F1A,JPT1,VSIR,TYROBP"
cGEP44,Fibroblast,Doublet,"COL1A1,COL1A2,COL3A1,SPARC,LUM,RARRES2,DCN,SFRP2,AEBP1,RPS26,BGN,IGFBP7,CALD1,COL6A3,ACTA2,TIMP3,COL6A2,COMP,TAGLN,CTGF,MYL9,MGP,CCDC80,MMP11,TPM2,CTHRC1,COL6A1,NBL1,THBS2,CXCL14"


In [5]:
#OUT_FILE <- "./3.1.Macro_GEP_Anno//3.3.Macro_cGEP_Anno_Complete_With_Genes.csv"
OUT_FILE <- '/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.6.cNMF_menchmark/cGEP_signature/Macro_cGEP77_signature.csv'

write.csv(final_combined_df, OUT_FILE, row.names = FALSE)